# 05 — ¿Por qué PatchCore rinde peor en FabriD? Diagnóstico

En RAW-FABRID (bloque B), PatchCore separa parches sanos y defectuosos con **AUROC 0,90**. En FabriD (notebook 04b), sobre el mismo test que el segmentador supervisado, cae a **0,64**, mientras YOLO11n-seg llega a **0,87**. Este notebook busca la causa.

**No entrena ni elige nada.** Lee los mapas de anomalía que guardaron el 04 (valid) y el 04b (test) y los desglosa. Ningún resultado de acá cambia modelos, umbrales ni configuraciones. Por eso se puede mirar test sin romper el protocolo: el test ya se evaluó una vez y sólo se describe.

| Prueba | Hipótesis que pone a prueba | Qué esperaríamos si fuera cierta |
|---|---|---|
| **1. Por sesión de captura y formato** | FabriD mezcla telas y condiciones de captura muy distintas. Para PatchCore, "lo normal" es demasiado amplio y un único banco y umbral no sirven para todas | AUROC alto **dentro** de cada sesión y bajo en el global |
| **2. Puntajes de las normales y peores casos** | Algunas telas sanas parecen anómalas para el banco | Normales de ciertas sesiones con puntajes altos; mapas encendidos en estampados, pliegues o bordes |
| **3. Por clase de defecto** | Algunas clases (hilo faltante, marca de aguja) casi no se distinguen de la textura | AUROC muy distinto entre clases |
| **4. Por tamaño de defecto** | PatchCore achica la imagen a 256×256 y pierde los defectos chicos (la misma pregunta de escala del bloque A) | AUROC y cobertura de regiones mucho peores en los defectos chicos |
| **5. Atajo de YOLO** | Parte de la ventaja de YOLO viene de reconocer la captura y no el defecto: 372 de las 467 marcas de aguja son originales 512×512 (imágenes cuadradas en el export) y ninguna normal lo es | YOLO cae mucho al quitar las imágenes cuadradas o las marcas de aguja |
| 6. Cantidad de normales (opcional, valid) | A PatchCore le faltan normales (429 contra 14.196 parches en RAW-FABRID) | El AUROC sigue subiendo con más normales |

> **Sesión de captura:** fecha del nombre del archivo (`IMG_20260714_...`). Según el EDA (FAB-2), las normales de mayo están fechadas el 05 y los defectos de esa tanda el 06; se tratan como una sola sesión, "mayo 05-06". Los defectos de **julio 22** no tienen ninguna normal en ningún split: PatchCore nunca vio esa tela sana.

**Datos que necesita** (Add Input en Kaggle; carpetas o ZIP, se detectan solos):

1. el Output del notebook **04** (validación completa, versión 3): carpeta `results/fabrid/comparacion04/`;
2. el Output del notebook **04b** (test): carpeta `results/fabrid/comparacion04_test/`. Sin él, se analiza sólo valid;
3. el dataset **`fabrid-v3-export-anotado`** (export con `train/valid/test`, `images/labels`).

No necesita GPU. La prueba 6 es la única que calcula algo nuevo (ajusta PatchCore en CPU) y está apagada por defecto.

## 0. Parámetros

In [ ]:
RAMA = "sofi"           # rama del repo con src/fabrid_diagnostico.py (Kaggle la clona)
RUTA_VALID = ""         # carpeta comparacion04 o ZIP del 04; vacío = detección automática
RUTA_TEST = ""          # carpeta comparacion04_test o ZIP del 04b; vacío = detección automática
RUTA_EXPORT = ""        # carpeta con train/valid/test del export FabriD; vacío = detección automática
REPLICAS_BOOTSTRAP = 1000
SEMILLA = 0
CASOS = 6               # imágenes por figura de peores casos

CORRER_CURVA_NORMALES = False       # prueba 6: ajusta PatchCore varias veces en CPU (≈ decenas de minutos)
N_NORMALES_CURVA = (50, 100, 200)   # además del banco completo de 429 que ya está guardado

## 1. Entorno y artefactos

En Kaggle se clona el repo en la rama `RAMA`; localmente se usa el repo donde está este notebook. El módulo `fabrid_comparison.py` se carga desde el Output del 04, el mismo que produjo los mapas.

In [ ]:
import sys, os, json, zipfile, subprocess, importlib.util, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EN_KAGGLE = Path("/kaggle/working").is_dir()
if EN_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "tp_final_vision_05"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--branch", RAMA,
                        "https://github.com/victoriapersico/tp_final_vision.git", str(REPO)], check=True)
    BUSCAR_EN = Path("/kaggle/input")
    OUT = WORK / "results/fabrid/diagnostico05"
else:
    REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
    BUSCAR_EN = REPO / "results"
    OUT = REPO / "results/fabrid/diagnostico05"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / "src"))
import fabrid_diagnostico as fd
from fabrid_diagnostico import MODELOS


def buscar_carpeta(ruta, nombre, marca):
    # Carpeta `nombre` que contiene `marca`, suelta o dentro de un ZIP (Kaggle puede montar cualquiera de los dos).
    raiz = Path(ruta) if ruta else BUSCAR_EN
    if raiz.is_dir() and raiz.name == nombre and (raiz / marca).is_file():
        return raiz
    candidatas = {p.parent for p in raiz.rglob(marca) if p.parent.name == nombre} if raiz.is_dir() else set()
    zips = [raiz] if raiz.suffix == ".zip" else (list(raiz.rglob("*.zip")) if raiz.is_dir() else [])
    for z in ([] if candidatas else zips):
        with zipfile.ZipFile(z) as archivo:
            dentro = [n for n in archivo.namelist() if n.endswith(f"{nombre}/{marca}")]
            if dentro:
                destino = Path("/tmp" if EN_KAGGLE else OUT) / f"_zip_{nombre}"
                if not destino.exists():
                    archivo.extractall(destino)
                candidatas.add((destino / dentro[0]).parent)
    if len(candidatas) > 1:
        raise RuntimeError(f"Hay más de una carpeta {nombre}: {sorted(candidatas)}. Indicar la ruta en los parámetros.")
    return candidatas.pop() if candidatas else None


VAL = buscar_carpeta(RUTA_VALID, "comparacion04", "metricas_valid.json")
TEST = buscar_carpeta(RUTA_TEST, "comparacion04_test", "metricas_test.json")
if VAL is None:
    raise FileNotFoundError("Falta el Output del notebook 04 (carpeta comparacion04). Agregarlo con Add Input.")
SPLITS = ["valid"] + (["test"] if TEST else [])
print("Validación:", VAL)
print("Test:", TEST or "no encontrado: se analiza sólo valid")

spec = importlib.util.spec_from_file_location("fabrid_comparison", VAL / "fabrid_comparison.py")
fc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fc)
config = json.loads((VAL / "configuracion.json").read_text())
metricas = {"valid": json.loads((VAL / "metricas_valid.json").read_text())}
if TEST:
    metricas["test"] = json.loads((TEST / "metricas_test.json").read_text())
UMBRALES = {m: float(metricas["valid"][m]["umbral_valid"]) for m in MODELOS}
print("Umbrales de valid (congelados):", UMBRALES)

In [ ]:
def es_export(path):
    return all((path / s / k).is_dir() for s in ("train", "valid", "test") for k in ("images", "labels"))

if RUTA_EXPORT:
    export = Path(RUTA_EXPORT)
else:
    raiz = BUSCAR_EN if EN_KAGGLE else Path(os.environ.get("REWEAR_DATA_DIR", REPO / "data")) / "fabrid_v3"
    candidatos = sorted({p.parent for p in raiz.rglob("train") if p.is_dir() and es_export(p.parent)})
    if len(candidatos) != 1:
        raise RuntimeError(f"Se esperaba UN export FabriD con train/valid/test; encontrados: {candidatos}")
    export = candidatos[0]
if not es_export(export):
    raise FileNotFoundError("El export requiere train/valid/test con images/labels")

# load_items espera <DATA>/fabrid_v3/annotations_export; si el export está en otro lado se enlaza.
if export.name == "annotations_export" and export.parent.name == "fabrid_v3":
    DATA = export.parent.parent
else:
    DATA = Path("/tmp/rewear_fabrid05") if EN_KAGGLE else OUT / "_data"
    enlace = DATA / "fabrid_v3/annotations_export"
    enlace.parent.mkdir(parents=True, exist_ok=True)
    if not enlace.exists():
        enlace.symlink_to(export.resolve(), target_is_directory=True)

manifest = REPO / "reports/fabrid/fab4_split_agrupado.csv"
if hashlib.sha256(manifest.read_bytes()).hexdigest() != config["manifiesto_sha256"]:
    raise ValueError("El manifiesto del repo no es el que usó el notebook 04")
items = fc.load_items(DATA, manifest)
por_id = {i["id"]: i for i in items}
evaluados = [i for i in items if i["split"] in SPLITS]
print({s: sum(i["split"] == s for i in items) for s in ("train", "valid", "test")})

## 2. Tabla por imagen y por región

Para cada imagen de valid y test: sesión, formato del export, clases presentes, área de defecto y el puntaje de cada modelo (máximo de su mapa, igual que en el 04). Para cada región defectuosa (componente conexa de la máscara común): clase, área y qué fracción de sus píxeles marca cada modelo con su umbral de valid.

Dos controles antes de analizar: las máscaras por clase tienen que reconstruir la máscara común del 04, y el AUROC imagen recalculado tiene que coincidir con el que guardaron el 04 y el 04b.

In [ ]:
for item in evaluados[:: max(1, len(evaluados) // 25)]:
    union = np.logical_or.reduce(list(fd.class_masks(item).values()))
    assert np.array_equal(union, fc.binary_mask(item)), item["id"]

mapas = {"valid": {m: fd.Mapas(VAL / m.replace("-", "_"), "valid") for m in MODELOS}}
if TEST:
    mapas["test"] = {m: fd.Mapas(TEST / m.replace("-", "_"), "test") for m in MODELOS}
for split in SPLITS:
    for m in MODELOS:
        ids = [i["id"] for i in evaluados if i["split"] == split]
        assert sorted(ids) == sorted(mapas[split][m].order), f"Las imágenes de {split}/{m} no coinciden"

imagenes, regiones = fd.build_tables(evaluados, mapas, UMBRALES)
for split in SPLITS:
    for m in MODELOS:
        recalculado = fd.auroc_bootstrap(imagenes[imagenes.split == split], f"score_{m}", n=0)[0]
        guardado = metricas[split][m]["AUROC_imagen"]
        assert abs(recalculado - guardado) < 1e-9, (split, m, recalculado, guardado)
print("Controles superados: máscaras y AUROC imagen coinciden con el 04/04b.")
imagenes.to_csv(OUT / "imagenes.csv", index=False)
regiones.to_csv(OUT / "regiones.csv", index=False)
pd.crosstab([imagenes.split, imagenes.sesion], imagenes.label.map({0: "normal", 1: "defectuosa"}))

In [ ]:
# Composición: qué tienen las normales y las defectuosas de cada formato. Las cuadradas son el posible atajo.
pd.crosstab([imagenes.label.map({0: "normal", 1: "defectuosa"}), imagenes.formato], imagenes.sesion, margins=True)

## 3. Prueba 1 — Dentro de cada sesión vs. global

Si el problema es que FabriD mezcla capturas, PatchCore debería separar bien sanas y defectuosas **dentro** de una misma sesión y fallar en el global, donde una tela sana de una sesión puede puntuar más que un defecto de otra.

`ambos` junta valid y test para tener más imágenes por grupo; los intervalos son bootstrap por `source_id`. Sólo se calcula el AUROC donde hay normales y defectuosas.

In [ ]:
def por_split(df):
    yield from ((s, df[df.split == s]) for s in SPLITS)
    if len(SPLITS) > 1:
        yield "ambos", df

tablas = []
for nombre, parte in por_split(imagenes):
    subsets = {"global": parte.label >= 0}
    subsets |= {f"sesión {s}": parte.sesion == s for s in sorted(parte.sesion.unique())}
    subsets |= {f"formato {f}": parte.formato == f for f in sorted(parte.formato.unique())}
    subsets["sin las sesiones sin normales"] = parte.sesion.isin(parte[parte.label == 0].sesion.unique())
    t = fd.auroc_table(parte, subsets, REPLICAS_BOOTSTRAP, SEMILLA)
    tablas.append(t.assign(split=nombre).set_index("split", append=True))
prueba1 = pd.concat(tablas).swaplevel().sort_index(level=0, sort_remaining=False)
prueba1.to_csv(OUT / "prueba1_sesion_formato.csv")
prueba1.round(3)

**Matriz cruzada.** Cada celda es el AUROC de las defectuosas de una sesión (columna) contra las normales de otra (fila), con valid y test juntos. Una fila baja indica que esas normales "parecen anómalas" para el banco. Una columna alta en todas las filas indica que esa sesión se distingue fácil, sea por el defecto o por la captura.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
for ax, m in zip(axes, MODELOS):
    matriz = fd.cross_auroc(imagenes, f"score_{m}")
    matriz.to_csv(OUT / f"prueba1_matriz_{m}.csv")
    im = ax.imshow(matriz.to_numpy(float), cmap="RdYlGn", vmin=0.3, vmax=1)
    ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(matriz.index)), matriz.index)
    for (r, c), v in np.ndenumerate(matriz.to_numpy(float)):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(m)
    ax.set_xlabel("defectuosas de")
    ax.set_ylabel("normales de")
fig.colorbar(im, ax=axes, shrink=0.8, label="AUROC")
plt.savefig(OUT / "prueba1_matriz.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Prueba 2 — ¿Qué puntaje reciben las telas sanas?

Distribución del puntaje de imagen de PatchCore (máximo del mapa) por sesión. La línea punteada es el umbral de píxel elegido en valid, como referencia de escala. Si las normales de una sesión quedan tan altas como los defectos de otra, ningún umbral único puede separarlas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
for ax, m in zip(axes, MODELOS):
    sesiones = sorted(imagenes.sesion.unique())
    for k, s in enumerate(sesiones):
        for desplazamiento, (label, color) in zip((-0.15, 0.15), ((0, "tab:blue"), (1, "tab:red"))):
            y = imagenes[(imagenes.sesion == s) & (imagenes.label == label)][f"score_{m}"]
            x = k + desplazamiento + np.random.default_rng(k).uniform(-0.08, 0.08, len(y))
            ax.scatter(x, y, s=6, alpha=0.5, color=color, label=("normal", "defectuosa")[label] if k == 0 else None)
    ax.axhline(UMBRALES[m], color="gray", ls=":", lw=1)
    ax.set_xticks(range(len(sesiones)), sesiones, rotation=20)
    ax.set_title(f"{m}: puntaje por imagen")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "prueba2_puntajes_por_sesion.png", dpi=130)
plt.show()
imagenes.groupby(["sesion", "label"])[[f"score_{m}" for m in MODELOS]].median().round(3)

**Peores casos de PatchCore.** Arriba, las normales con mayor puntaje (falsas alarmas más fuertes). Abajo, las defectuosas con menor puntaje (defectos que no ve). El mapa usa una escala común (hasta 1,5 veces el umbral) y el contorno verde marca el defecto real. Conviene mirar si las normales se encienden en estampados, pliegues, bordes o cambios de iluminación, y si los defectos perdidos son chicos o de bajo contraste.

In [ ]:
def peores(df, label, ascending):
    return df[df.label == label].sort_values("score_PatchCore", ascending=ascending).head(CASOS)

fig, axes = plt.subplots(2, CASOS, figsize=(2.6 * CASOS, 6))
for fila, casos in enumerate((peores(imagenes, 0, False), peores(imagenes, 1, True))):
    for ax, (_, r) in zip(axes[fila], casos.iterrows()):
        item = por_id[r.id]
        mascara = fc.binary_mask(item) if r.label else None
        fd.overlay(ax, item, mapas[r.split]["PatchCore"][r.id], mascara, vmax=1.5 * UMBRALES["PatchCore"],
                   title=f"{r.sesion} · {r.formato}\n{r.clases or 'normal'} · {r.score_PatchCore:.2f}")
    for ax in axes[fila][len(casos):]:
        ax.axis("off")
fig.suptitle("PatchCore. Arriba: normales con mayor puntaje. Abajo: defectuosas con menor puntaje "
             "(contorno verde = defecto real)", fontsize=10)
plt.tight_layout()
plt.savefig(OUT / "prueba2_peores_casos_patchcore.png", dpi=130)
plt.show()

## 5. Prueba 3 — Por clase de defecto

AUROC de cada clase contra **todas** las normales del split. Una imagen con dos clases cuenta en las dos. La cobertura de regiones mide cuánto de cada defecto real marca el mapa con el umbral de valid.

In [ ]:
tablas = []
for nombre, parte in por_split(imagenes):
    subsets = {c: (parte.label == 0) | parte.clases.str.contains(c, regex=False) for c in fd.CLASES}
    tablas.append(fd.auroc_table(parte, subsets, REPLICAS_BOOTSTRAP, SEMILLA).assign(split=nombre)
                  .set_index("split", append=True))
prueba3 = pd.concat(tablas).swaplevel().sort_index(level=0, sort_remaining=False)
prueba3.to_csv(OUT / "prueba3_clase.csv")
display(prueba3.round(3))
cobertura_clase = fd.coverage_table(regiones, "clase")
cobertura_clase.to_csv(OUT / "prueba3_cobertura_clase.csv")
cobertura_clase.round(3)

## 6. Prueba 4 — Por tamaño de defecto

PatchCore recibe cada imagen achicada y deformada a 256×256: una foto 640×360 pierde mucho más alto que una 640×640. Si los defectos chicos son los que se pierden, es la misma pregunta de resolución del bloque A.

El tamaño es el área de defecto como porcentaje de la imagen. Para el AUROC imagen se usa el área total de defecto de cada imagen; para la cobertura, el área de cada región.

In [ ]:
tablas = []
for nombre, parte in por_split(imagenes):
    subsets = {t: (parte.label == 0) | (parte.tamano == t) for t in fd.TAMANOS_ETIQUETAS}
    tablas.append(fd.auroc_table(parte, subsets, REPLICAS_BOOTSTRAP, SEMILLA).assign(split=nombre)
                  .set_index("split", append=True))
prueba4 = pd.concat(tablas).swaplevel().sort_index(level=0, sort_remaining=False)
prueba4.to_csv(OUT / "prueba4_tamano.csv")
display(prueba4.round(3))
cobertura_tamano = fd.coverage_table(regiones, "tamano")
cobertura_tamano.to_csv(OUT / "prueba4_cobertura_tamano.csv")
display(cobertura_tamano.round(3))

fig, ax = plt.subplots(figsize=(6, 3.5))
for m, color in zip(MODELOS, ("tab:green", "tab:purple")):
    ax.plot(cobertura_tamano.index.astype(str), cobertura_tamano[f"cubierta media {m}"], "o-", color=color, label=m)
ax.set_xlabel("área de la región (% de la imagen)")
ax.set_ylabel("fracción media cubierta")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "prueba4_cobertura_tamano.png", dpi=130)
plt.show()

Cobertura por formato: si PatchCore pierde más en las imágenes apaisadas, la deformación a cuadrado pesa.

In [ ]:
cobertura_formato = fd.coverage_table(regiones, "formato")
cobertura_formato.to_csv(OUT / "prueba4_cobertura_formato.csv")
cobertura_formato.round(3)

## 7. Prueba 5 — ¿YOLO usa un atajo de captura?

Se recalculan las métricas del 04/04b sacando, por separado, las imágenes cuadradas (casi todas marcas de aguja de originales 512×512, un formato que ninguna normal tiene) y las imágenes con marca de aguja. Si YOLO cae mucho, parte de su ventaja era reconocer la captura y no el defecto. También se repite PatchCore para ver si la brecha entre los dos se mantiene.

Las métricas por píxel usan la misma función y el mismo umbral de valid que el 04.

In [ ]:
filtros = {
    "todas": lambda r: True,
    "sin imágenes cuadradas": lambda r: r["formato"] != "cuadrada",
    "sin marcas de aguja": lambda r: "Needle Mark" not in r["clases"],
    "sin cuadradas ni marcas de aguja": lambda r: r["formato"] != "cuadrada" and "Needle Mark" not in r["clases"],
}
filas = []
for split in SPLITS:
    tabla = imagenes[imagenes.split == split]
    for nombre, filtro in filtros.items():
        ids = [r["id"] for _, r in tabla.iterrows() if filtro(r)]
        sub = [por_id[i] for i in ids]
        for m in MODELOS:
            res = fc.evaluate_maps(sub, [mapas[split][m][i] for i in ids], UMBRALES[m], confirm_test=split == "test")
            lo, hi = fd.auroc_bootstrap(tabla[tabla.id.isin(ids)], f"score_{m}", REPLICAS_BOOTSTRAP, SEMILLA)[1]
            filas.append({"split": split, "subconjunto": nombre, "modelo": m,
                          "normales": sum(i["label"] == 0 for i in sub), "defectuosas": sum(i["label"] for i in sub),
                          "AUROC imagen": res["AUROC_imagen"], "IC95 imagen": f"{lo:.2f}–{hi:.2f}",
                          "AUROC píxel": res["AUROC_pixel"], "AUPRO@0.3": res["AUPRO@0.3"], "Dice": res["Dice_micro"]})
prueba5 = pd.DataFrame(filas).set_index(["split", "subconjunto", "modelo"])
prueba5.to_csv(OUT / "prueba5_atajo_yolo.csv")
prueba5.round(3)

## 8. Prueba 6 (opcional) — ¿Cuántas normales necesita PatchCore?

Sólo con `CORRER_CURVA_NORMALES = True`. Ajusta PatchCore con la misma configuración del 04 (ResNet18, 256, coreset 1 %) usando subconjuntos de las 429 normales de train, elegidos con semilla, y evalúa **sólo en valid**. El punto de 429 es el banco ya guardado, así que no se reajusta. Si la curva sigue subiendo al final, faltan normales; si se aplana, el límite está en otro lado.

In [ ]:
if not CORRER_CURVA_NORMALES:
    print("Prueba 6 apagada (CORRER_CURVA_NORMALES = False).")
else:
    import time
    import anomaly_baseline as ab
    valid = [i for i in items if i["split"] == "valid"]
    normales = sorted((i for i in items if i["split"] == "train" and i["label"] == 0), key=lambda i: i["id"])
    sesiones_valid = imagenes[imagenes.split == "valid"].set_index("id").sesion
    curva = []
    for n in (*N_NORMALES_CURVA, len(normales)):
        inicio = time.perf_counter()
        if n == len(normales):
            pc = ab.PatchCore(backbone=config["pc"]["backbone"], size=config["pc"]["size"],
                              coreset=config["pc"]["coreset"], seed=config["semilla"])
            with np.load(VAL / "patchcore_memoria.npz") as guardado:
                pc.state = {k: guardado[k] for k in guardado.files}
        else:
            elegidas = [normales[int(k)] for k in np.random.default_rng(SEMILLA).choice(len(normales), n, replace=False)]
            pc = fc.fit_patchcore(elegidas, **config["pc"], seed=config["semilla"])
        maps_n = fc.predict_patchcore(pc, valid)
        res = fc.evaluate_maps(valid, maps_n, fc.choose_threshold(valid, maps_n))
        puntajes = pd.DataFrame({"label": [i["label"] for i in valid], "source_id": [i["source_id"] for i in valid],
                                 "sesion": [sesiones_valid[i["id"]] for i in valid],
                                 "score": [float(x.max()) for x in maps_n]})
        dentro = [fd.auroc_bootstrap(g, "score", n=0)[0] for _, g in puntajes.groupby("sesion")]
        curva.append({"normales": n, "AUROC imagen": res["AUROC_imagen"], "AUROC píxel": res["AUROC_pixel"],
                      "AUPRO@0.3": res["AUPRO@0.3"], "AUROC medio dentro de sesión": np.nanmean(dentro),
                      "segundos": time.perf_counter() - inicio})
        print(curva[-1])
    curva = pd.DataFrame(curva).set_index("normales")
    curva.to_csv(OUT / "prueba6_curva_normales.csv")
    display(curva.round(3))

## 9. Guardar

In [ ]:
if EN_KAGGLE:
    import shutil
    print(shutil.make_archive(str(WORK / "results_diagnostico05"), "zip", root_dir=WORK,
                              base_dir="results/fabrid/diagnostico05"))
print(sorted(p.name for p in OUT.iterdir() if not p.name.startswith("_")))

## 10. Conclusiones

Resultados de la corrida en Kaggle con los Outputs del 04 (validación v3) y del 04b, sin GPU. Los controles se superaron: máscaras y AUROC imagen coinciden con lo reportado. "Ambos" es valid + test juntos, sólo para describir.

**En una frase:** en FabriD, PatchCore localiza en parte (AUROC píxel 0,74 en valid), pero la tela sana genera puntajes tan altos como los defectos, así que el puntaje de imagen casi no separa. La caída no se explica por la mezcla de sesiones, el formato ni el tamaño.

- **Prueba 1, sesión y formato.** Dentro de cada sesión, PatchCore no mejora de forma clara respecto del global (0,62 en ambos): da 0,46 en agosto 03, 0,61 en julio 14 y 0,76 en mayo 05-06. YOLO11n-seg se mantiene entre 0,88 y 0,93 en todas. La hipótesis de que "lo normal" es demasiado amplio entre sesiones **no alcanza** para explicar la caída. En la matriz cruzada de PatchCore manda la fila, no la columna:
  - las normales de **agosto 03** dan entre 0,44 y 0,53 contra los defectos de cualquier sesión;
  - las de julio 14, entre 0,50 y 0,61;
  - las de mayo 05-06, entre 0,65 y 0,78.

  Es decir, el problema está en cuánto puntúan las telas sanas, no en que los defectos de una sesión se reconozcan por la captura. Por formato, las imágenes cuadradas son las peores para PatchCore (0,54, IC 0,40–0,68); YOLO no cambia.
- **Prueba 2, puntajes.** La media del puntaje de imagen (máximo del mapa) de las normales es prácticamente igual a la de las defectuosas:
  - agosto 03: 2,99 las normales contra 2,93 las defectuosas (las sanas puntúan más);
  - julio 14: 2,86 contra 2,92;
  - mayo 05-06: 2,70 contra 2,92.

  El máximo del mapa cae casi siempre en algún punto del fondo, así que ningún umbral único separa. YOLO, en cambio, deja las normales en 0,02–0,05 y las defectuosas en 0,24–0,65. Los peores casos (figura de la prueba 2) quedan para revisar a ojo qué patrones de fondo se encienden (estampados, pliegues o bordes).
- **Prueba 3, clase.** PatchCore no funciona bien en ninguna clase (AUROC de 0,58 a 0,70 en ambos splits). Cobertura media de cada región y % de regiones que no toca:
  - mancha: 0,54 y 33 % sin tocar;
  - agujero: 0,31 y 57 %;
  - marca de aguja: 0,21 y 58 %;
  - hilo faltante: 0,14 y 63 %.

  Las manchas, que cambian el color, son lo único que el banco de parches detecta de forma razonable. Los defectos de estructura del tejido se pierden. YOLO también tiene más dificultad con hilo faltante (cobertura 0,44) y marca de aguja (0,49), pero los encuentra mucho más.
- **Prueba 4, tamaño.** Los defectos de menos de 0,1 % del área son los más difíciles para los dos modelos: PatchCore no toca el 79 % y YOLO el 63 %. Pero el tamaño no explica la brecha: incluso con defectos de 2 % o más, el AUROC de PatchCore es 0,66 contra 0,91 de YOLO. Por formato, PatchCore cubre peor las cuadradas (0,21 contra 0,33–0,36).
- **Prueba 5, atajo de YOLO.** Sacar las imágenes cuadradas y las marcas de aguja casi no mueve a YOLO: en valid pasa de 0,939 a 0,941 de AUROC imagen y en test de 0,874 a 0,875. Su AUPRO incluso sube (de 0,77 a 0,86 en valid). **YOLO no depende de ese atajo:** su ventaja es real. PatchCore se queda en 0,62–0,64.
- **Prueba 6:** no se corrió (`CORRER_CURVA_NORMALES = False`). Queda abierto si más normales mejorarían a PatchCore; dado el resultado de la prueba 2, lo esperable es una mejora limitada.
- **Para el informe:** la tabla de la prueba 1 (global y por sesión) o la dispersión de la prueba 2, más una frase sobre la prueba 5: "sin imágenes cuadradas ni marcas de aguja, YOLO11n-seg mantiene AUROC 0,94 en valid; la ventaja no es un atajo de formato". Lo demás queda como respaldo para la defensa.
- **Limitaciones:**
  - pocos casos en agosto 03 (30 normales y 36 defectuosas entre los dos splits), con intervalos anchos;
  - julio 22 no tiene normales, así que no tiene AUROC propio;
  - la sesión se infiere de la fecha del nombre del archivo;
  - valid y test juntos sólo se usan para describir: nada de este notebook cambia modelos ni umbrales.